# 4. RAG Ingestion：怎样用版本、幂等键与 tombstone 防止重复和旧文档回流？

## 面试回答主线

RAG ingestion 不是把文件切块后无限 append，而是一条有版本的状态机。每个事件应有唯一 event_id，每篇文档维护最高 version，chunk_id 由 doc_id、version、序号与内容哈希确定；重复事件重放应得到相同结果，旧版本事件必须拒绝。更新时新版本取代旧 active chunks，删除则写 tombstone 并从可检索视图移除。面试时我会对真实政策文档执行 create、duplicate、update、stale replay 与 delete，比较朴素 append 基线和幂等状态机。还要真实复现“写入一半进程崩溃后重试”如何制造重复，并用 deterministic upsert 修复。生产系统需要 outbox/WAL、对象存储版本、embedding 版本与 ACL 同步。

## 1. 真实案例：六篇业务文档及乱序、重复、更新和删除事件

文档覆盖退款、发票、数据库、安全、RAG 引用和配送。事件流包含同 event_id 重投、v2 更新后 v1 迟到，以及文档删除；内容用句号切成可读语义 chunk。

In [1]:
from pprint import pprint  # 导入结构化打印工具展示 ingestion 事件与索引状态
import hashlib  # 导入哈希函数生成确定性 chunk_id 和内容校验值
events = [{"event_id": "E01", "doc_id": "DOC-REFUND", "version": 1, "op": "upsert", "content": "未发货订单七天内可退款。退款原路返回。"}, {"event_id": "E02", "doc_id": "DOC-INVOICE", "version": 1, "op": "upsert", "content": "发票开具前可修改抬头。开票后需财务审核。"}, {"event_id": "E03", "doc_id": "DOC-DB", "version": 1, "op": "upsert", "content": "慢查询先检查执行计划。再查看索引命中。"}, {"event_id": "E04", "doc_id": "DOC-SECURITY", "version": 1, "op": "upsert", "content": "发现密钥泄露应立即轮换。随后审计访问日志。"}, {"event_id": "E05", "doc_id": "DOC-RAG", "version": 1, "op": "upsert", "content": "RAG 回答应附来源。引用必须指向原文。"}, {"event_id": "E06", "doc_id": "DOC-SHIP", "version": 1, "op": "upsert", "content": "同城订单次日送达。暴雨可能顺延一天。"}, {"event_id": "E01", "doc_id": "DOC-REFUND", "version": 1, "op": "upsert", "content": "未发货订单七天内可退款。退款原路返回。"}, {"event_id": "E07", "doc_id": "DOC-REFUND", "version": 2, "op": "upsert", "content": "未发货订单三天内可退款。退款原路返回。"}, {"event_id": "E08", "doc_id": "DOC-REFUND", "version": 1, "op": "upsert", "content": "未发货订单七天内可退款。退款原路返回。"}, {"event_id": "E09", "doc_id": "DOC-SHIP", "version": 2, "op": "delete", "content": ""}]  # 定义六文档对应的十个真实事件并注入重复、更新、迟到和删除
def split_sentences(content):  # 手写中文句号切块以暴露真实 chunk 内容
    return [sentence.strip() for sentence in content.split("。") if sentence.strip()]  # 去除空句并保留可检索语义片段
preview = [{"event": event["event_id"], "doc": event["doc_id"], "version": event["version"], "op": event["op"], "chunks": split_sentences(event["content"])} for event in events]  # 汇总 ingestion 的关键事件字段
print("RAG ingestion 事件预览：")  # 输出真实案例标题
pprint(preview, sort_dicts=False)  # 展示乱序和重复事件中包含的真实文档内容

RAG ingestion 事件预览：
[{'event': 'E01',
  'doc': 'DOC-REFUND',
  'version': 1,
  'op': 'upsert',
  'chunks': ['未发货订单七天内可退款', '退款原路返回']},
 {'event': 'E02',
  'doc': 'DOC-INVOICE',
  'version': 1,
  'op': 'upsert',
  'chunks': ['发票开具前可修改抬头', '开票后需财务审核']},
 {'event': 'E03',
  'doc': 'DOC-DB',
  'version': 1,
  'op': 'upsert',
  'chunks': ['慢查询先检查执行计划', '再查看索引命中']},
 {'event': 'E04',
  'doc': 'DOC-SECURITY',
  'version': 1,
  'op': 'upsert',
  'chunks': ['发现密钥泄露应立即轮换', '随后审计访问日志']},
 {'event': 'E05',
  'doc': 'DOC-RAG',
  'version': 1,
  'op': 'upsert',
  'chunks': ['RAG 回答应附来源', '引用必须指向原文']},
 {'event': 'E06',
  'doc': 'DOC-SHIP',
  'version': 1,
  'op': 'upsert',
  'chunks': ['同城订单次日送达', '暴雨可能顺延一天']},
 {'event': 'E01',
  'doc': 'DOC-REFUND',
  'version': 1,
  'op': 'upsert',
  'chunks': ['未发货订单七天内可退款', '退款原路返回']},
 {'event': 'E07',
  'doc': 'DOC-REFUND',
  'version': 2,
  'op': 'upsert',
  'chunks': ['未发货订单三天内可退款', '退款原路返回']},
 {'event': 'E08',
  'doc': 'DOC-REFUND',
  'version': 1,
  'op'

## 2. Baseline（基线）：每次事件都向向量索引 append chunks

朴素消费者不检查 event_id、version 或 delete，收到 upsert 就追加。重复 E01、更新 v2 与迟到 v1 会让退款索引同时包含七天和三天；delete 也没有清除配送文档。

In [2]:
def naive_ingest(event_stream):  # 实现没有版本和幂等控制的错误 append 消费者
    index = []  # 用列表模拟只增不改的向量索引
    for event in event_stream:  # 按消息队列到达顺序处理全部事件
        if event["op"] == "upsert":  # 朴素消费者只认识新增内容
            for position, text in enumerate(split_sentences(event["content"])):  # 对当前事件正文重新切块
                index.append({"doc_id": event["doc_id"], "version": event["version"], "position": position, "text": text, "event_id": event["event_id"]})  # 无条件追加导致重复与旧版本并存
    return index  # 返回包含历史垃圾的错误可检索索引
naive_index = naive_ingest(events)  # 在完整乱序事件流上运行 append 基线
naive_refund_chunks = [chunk for chunk in naive_index if chunk["doc_id"] == "DOC-REFUND"]  # 提取退款文档观察冲突版本
naive_ship_chunks = [chunk for chunk in naive_index if chunk["doc_id"] == "DOC-SHIP"]  # 提取已经收到 delete 的配送文档
print("朴素 append 后的退款 chunks：")  # 输出错误基线的版本冲突标题
pprint(naive_refund_chunks, sort_dicts=False)  # 展示重复 v1、v2 与迟到 v1 同时可检索
print({"朴素索引总chunk数": len(naive_index), "退款chunk数": len(naive_refund_chunks), "删除后配送仍可检索": len(naive_ship_chunks) > 0})  # 汇总基线的重复和删除失效问题

朴素 append 后的退款 chunks：
[{'doc_id': 'DOC-REFUND',
  'version': 1,
  'position': 0,
  'text': '未发货订单七天内可退款',
  'event_id': 'E01'},
 {'doc_id': 'DOC-REFUND',
  'version': 1,
  'position': 1,
  'text': '退款原路返回',
  'event_id': 'E01'},
 {'doc_id': 'DOC-REFUND',
  'version': 1,
  'position': 0,
  'text': '未发货订单七天内可退款',
  'event_id': 'E01'},
 {'doc_id': 'DOC-REFUND',
  'version': 1,
  'position': 1,
  'text': '退款原路返回',
  'event_id': 'E01'},
 {'doc_id': 'DOC-REFUND',
  'version': 2,
  'position': 0,
  'text': '未发货订单三天内可退款',
  'event_id': 'E07'},
 {'doc_id': 'DOC-REFUND',
  'version': 2,
  'position': 1,
  'text': '退款原路返回',
  'event_id': 'E07'},
 {'doc_id': 'DOC-REFUND',
  'version': 1,
  'position': 0,
  'text': '未发货订单七天内可退款',
  'event_id': 'E08'},
 {'doc_id': 'DOC-REFUND',
  'version': 1,
  'position': 1,
  'text': '退款原路返回',
  'event_id': 'E08'}]
{'朴素索引总chunk数': 18, '退款chunk数': 8, '删除后配送仍可检索': True}


## 3. 手写核心状态机：事件幂等、单调版本与确定性 chunk_id

消费者先检查 event_id 是否处理过，再比较文档最高 version。有效 upsert 用确定性 ID 覆盖写入并移除旧 active 版本；delete 清空 active chunks 并记录 tombstone。处理结果 ledger 保留 applied、duplicate、stale 和 deleted。

In [3]:
def deterministic_chunk_id(doc_id, version, position, text):  # 为同一文档版本的切块生成可重放标识
    payload = f"{doc_id}|{version}|{position}|{text}"  # 组合稳定业务主键、版本、位置和内容
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]  # 截取内容哈希作为确定性 chunk_id
def ingest_idempotently(event_stream):  # 实现具有版本和删除语义的 ingestion 状态机
    processed_events = set()  # 记录已提交 event_id 防止消息重复生效
    document_state = {}  # 保存每篇文档的最高版本、删除状态和 active chunk IDs
    chunk_store = {}  # 用 chunk_id 字典模拟可幂等 upsert 的向量存储
    ledger = []  # 记录每个输入事件的处理决策与状态变化
    for event in event_stream:  # 按真实消息到达顺序消费事件
        if event["event_id"] in processed_events:  # 先检查消息级幂等键是否已经提交
            ledger.append({"event": event["event_id"], "doc": event["doc_id"], "decision": "duplicate_event"})  # 记录重复消息但不再次写索引
            continue  # 跳过已经完成的事件
        state = document_state.get(event["doc_id"], {"version": 0, "deleted": False, "chunks": []})  # 读取文档当前最高版本状态
        if event["version"] <= state["version"]:  # 检查迟到或重复版本是否会覆盖新知识
            processed_events.add(event["event_id"])  # 记录该旧事件已经被明确消费
            ledger.append({"event": event["event_id"], "doc": event["doc_id"], "decision": "stale_version", "current": state["version"]})  # 保存拒绝原因和当前水位
            continue  # 不允许旧版本重新进入 active 索引
        for old_chunk_id in state["chunks"]:  # 遍历上一个 active 版本的全部切块
            chunk_store.pop(old_chunk_id, None)  # 从可检索视图移除被新版本取代的旧 chunk
        if event["op"] == "delete":  # 单独处理文档删除事件
            document_state[event["doc_id"]] = {"version": event["version"], "deleted": True, "chunks": []}  # 写入带版本的 tombstone
            decision = "deleted"  # 标记该事件已生效为删除
        else:  # upsert 事件需要生成新版本 chunks
            new_chunk_ids = []  # 收集新 active 版本的确定性 chunk IDs
            for position, text in enumerate(split_sentences(event["content"])):  # 手写切分当前版本正文
                chunk_id = deterministic_chunk_id(event["doc_id"], event["version"], position, text)  # 根据内容与版本生成稳定主键
                chunk_store[chunk_id] = {"chunk_id": chunk_id, "doc_id": event["doc_id"], "version": event["version"], "position": position, "text": text}  # 以 upsert 语义写入向量存储
                new_chunk_ids.append(chunk_id)  # 保存该文档当前 active chunk 引用
            document_state[event["doc_id"]] = {"version": event["version"], "deleted": False, "chunks": new_chunk_ids}  # 原子替换文档版本视图
            decision = "upserted"  # 标记有效新版本已经生效
        processed_events.add(event["event_id"])  # 在状态写入成功后提交事件幂等键
        ledger.append({"event": event["event_id"], "doc": event["doc_id"], "decision": decision, "version": event["version"]})  # 保存可审计处理结果
    return document_state, chunk_store, ledger  # 返回文档水位、active 索引和事件账本
document_state, chunk_store, ingestion_ledger = ingest_idempotently(events)  # 在相同事件流上运行安全状态机
print("Ingestion 状态机逐事件账本：")  # 输出核心机制中间量标题
pprint(ingestion_ledger, sort_dicts=False)  # 展示 duplicate、stale、upsert 和 delete 决策

Ingestion 状态机逐事件账本：
[{'event': 'E01', 'doc': 'DOC-REFUND', 'decision': 'upserted', 'version': 1},
 {'event': 'E02', 'doc': 'DOC-INVOICE', 'decision': 'upserted', 'version': 1},
 {'event': 'E03', 'doc': 'DOC-DB', 'decision': 'upserted', 'version': 1},
 {'event': 'E04', 'doc': 'DOC-SECURITY', 'decision': 'upserted', 'version': 1},
 {'event': 'E05', 'doc': 'DOC-RAG', 'decision': 'upserted', 'version': 1},
 {'event': 'E06', 'doc': 'DOC-SHIP', 'decision': 'upserted', 'version': 1},
 {'event': 'E01', 'doc': 'DOC-REFUND', 'decision': 'duplicate_event'},
 {'event': 'E07', 'doc': 'DOC-REFUND', 'decision': 'upserted', 'version': 2},
 {'event': 'E08',
  'doc': 'DOC-REFUND',
  'decision': 'stale_version',
  'current': 2},
 {'event': 'E09', 'doc': 'DOC-SHIP', 'decision': 'deleted', 'version': 2}]


## 4. Active 索引结果：每篇文档只保留最高有效版本

退款只应保留 v2“三天内”，配送应因 v2 tombstone 完全不可检索。下面按 doc_id 汇总当前版本、删除状态和 active chunk 文本。

In [4]:
active_rows = []  # 收集六篇文档的最终版本化索引状态
for doc_id in sorted({event["doc_id"] for event in events}):  # 遍历事件流涉及的每篇真实文档
    state = document_state[doc_id]  # 读取文档最高版本与 tombstone 状态
    active_texts = [chunk_store[chunk_id]["text"] for chunk_id in state["chunks"]]  # 解析当前版本仍可检索的 chunk 内容
    active_rows.append({"doc": doc_id, "version": state["version"], "deleted": state["deleted"], "active_chunk数": len(active_texts), "active文本": active_texts})  # 保存逐文档可观察状态
refund_active = next(row for row in active_rows if row["doc"] == "DOC-REFUND")  # 取得退款政策最终 active 版本
ship_active = next(row for row in active_rows if row["doc"] == "DOC-SHIP")  # 取得配送文档最终 tombstone 状态
print("版本化 ingestion 的文档结果：")  # 输出核心方案结果标题
pprint(active_rows, sort_dicts=False)  # 展示六篇文档只保留最高有效版本

版本化 ingestion 的文档结果：
[{'doc': 'DOC-DB',
  'version': 1,
  'deleted': False,
  'active_chunk数': 2,
  'active文本': ['慢查询先检查执行计划', '再查看索引命中']},
 {'doc': 'DOC-INVOICE',
  'version': 1,
  'deleted': False,
  'active_chunk数': 2,
  'active文本': ['发票开具前可修改抬头', '开票后需财务审核']},
 {'doc': 'DOC-RAG',
  'version': 1,
  'deleted': False,
  'active_chunk数': 2,
  'active文本': ['RAG 回答应附来源', '引用必须指向原文']},
 {'doc': 'DOC-REFUND',
  'version': 2,
  'deleted': False,
  'active_chunk数': 2,
  'active文本': ['未发货订单三天内可退款', '退款原路返回']},
 {'doc': 'DOC-SECURITY',
  'version': 1,
  'deleted': False,
  'active_chunk数': 2,
  'active文本': ['发现密钥泄露应立即轮换', '随后审计访问日志']},
 {'doc': 'DOC-SHIP',
  'version': 2,
  'deleted': True,
  'active_chunk数': 0,
  'active文本': []}]


## 5. 结果解读：同一检索词不再返回相互冲突的版本

用简单子串检索“退款”时，朴素索引返回多个七天与三天事实；active 索引只返回 v2。这个检索器不是为了替代 BM25，而是把 ingestion 垃圾对 RAG 上下文的直接影响展示出来。

In [5]:
def keyword_search(index_chunks, keyword):  # 用可解释子串匹配观察 ingestion 对检索上下文的影响
    return [chunk for chunk in index_chunks if keyword in chunk["text"]]  # 返回正文中包含查询词的所有 chunks
naive_refund_hits = keyword_search(naive_index, "退款")  # 在朴素 append 索引中查询退款事实
active_refund_hits = keyword_search(list(chunk_store.values()), "退款")  # 在版本化 active 索引中查询退款事实
comparison = {"朴素命中文本": [f'v{chunk["version"]}:{chunk["text"]}' for chunk in naive_refund_hits], "幂等命中文本": [f'v{chunk["version"]}:{chunk["text"]}' for chunk in active_refund_hits], "朴素冲突版本": sorted(set(chunk["version"] for chunk in naive_refund_hits)), "active版本": sorted(set(chunk["version"] for chunk in active_refund_hits))}  # 汇总同一关键词的上下文差异
print("退款检索上下文对照：")  # 输出结果解读标题
pprint(comparison, sort_dicts=False)  # 展示重复旧知识如何污染 RAG prompt

退款检索上下文对照：
{'朴素命中文本': ['v1:未发货订单七天内可退款',
            'v1:退款原路返回',
            'v1:未发货订单七天内可退款',
            'v1:退款原路返回',
            'v2:未发货订单三天内可退款',
            'v2:退款原路返回',
            'v1:未发货订单七天内可退款',
            'v1:退款原路返回'],
 '幂等命中文本': ['v2:未发货订单三天内可退款', 'v2:退款原路返回'],
 '朴素冲突版本': [1, 2],
 'active版本': [2]}


## 6. 失败案例与修正：写入第一块后崩溃，重试导致重复

若消费者先 append 第一块、进程崩溃，再重放整条事件，列表索引会有三个条目且第一块重复。确定性 chunk_id 的 upsert 即使重复执行也只有两个键；幂等键应在数据写成功后提交，并通过事务/outbox 处理跨系统原子性。

In [6]:
crash_event = events[2]  # 选择包含两个句子的数据库文档复现部分写入崩溃
crash_chunks = split_sentences(crash_event["content"])  # 获取该事件应写入的两个真实 chunks
broken_store = []  # 用列表模拟不支持幂等 upsert 的向量存储
broken_store.append({"position": 0, "text": crash_chunks[0]})  # 模拟进程在写入第一块后崩溃
for position, text in enumerate(crash_chunks):  # 重启后从头重放完整事件
    broken_store.append({"position": position, "text": text})  # append 再次写入第一块造成重复
fixed_store = {}  # 用确定性 chunk_id 字典模拟安全 upsert 存储
fixed_store[deterministic_chunk_id(crash_event["doc_id"], crash_event["version"], 0, crash_chunks[0])] = crash_chunks[0]  # 模拟崩溃前已写入第一块
for position, text in enumerate(crash_chunks):  # 重启后同样重放完整事件
    chunk_id = deterministic_chunk_id(crash_event["doc_id"], crash_event["version"], position, text)  # 为每块重新计算相同稳定主键
    fixed_store[chunk_id] = text  # upsert 覆盖已有第一块并新增第二块
print({"失败_append条目数": len(broken_store), "失败内容": broken_store, "修正_upsert条目数": len(fixed_store), "修正内容": list(fixed_store.values())})  # 展示部分失败重试与幂等修复

{'失败_append条目数': 3, '失败内容': [{'position': 0, 'text': '慢查询先检查执行计划'}, {'position': 0, 'text': '慢查询先检查执行计划'}, {'position': 1, 'text': '再查看索引命中'}], '修正_upsert条目数': 2, '修正内容': ['慢查询先检查执行计划', '再查看索引命中']}


## 7. 生产差距与最小回归检查

真实 ingestion 还需协调消息队列 offset、对象存储、关系元数据和向量库，通常用 outbox/WAL 加最终一致性补偿。chunk_id 应包含 chunker 与 embedding 版本；ACL、语言、来源和 checksum 也要进入元数据。删除必须传播到缓存与副本，并保留审计 tombstone。下面的断言只验证本实验中的六文档、重复事件、版本、删除和 crash retry。

In [7]:
assert len({event["doc_id"] for event in events}) >= 6  # 确认真实业务文档数量满足逐文档教学要求
assert any(row["decision"] == "duplicate_event" for row in ingestion_ledger)  # 确认重复消息被幂等账本识别
assert any(row["decision"] == "stale_version" for row in ingestion_ledger)  # 确认迟到旧版本没有覆盖新知识
assert refund_active["version"] == 2 and "三天内" in refund_active["active文本"][0]  # 确认退款 active 索引只保留最新政策
assert ship_active["deleted"] is True and ship_active["active_chunk数"] == 0  # 确认 tombstone 从可检索视图移除配送文档
assert len(naive_refund_hits) > len(active_refund_hits)  # 确认朴素 append 真实制造重复检索上下文
assert len(broken_store) == 3 and len(fixed_store) == 2  # 确认部分崩溃重试的重复与确定性 upsert 修复
print("回归检查通过：事件幂等、版本水位、tombstone 与部分失败重试均已验证。")  # 输出最终验收结论

回归检查通过：事件幂等、版本水位、tombstone 与部分失败重试均已验证。
